# 01 - Normalizar RGB por streaming

Ejecuta la normalizacion de imagenes usando `normalizar_rgb_streaming.py` en el ambiente Python de rasterio. La entrada es una carpeta raiz; el script busca TIFF recursivamente y deja las salidas con el mismo nombre dentro de `normalizadas`.

In [ ]:
from pathlib import Path
import json
import os
import subprocess
import sys

def find_project_root(start=None):
    current = Path(start or Path.cwd()).resolve()
    for candidate in [current, *current.parents]:
        if (candidate / 'core').exists() and (candidate / 'flujo_geosupport_final' / 'settings.json').exists():
            return candidate
        if candidate.name.lower() == 'flujo_geosupport_final' and (candidate / 'settings.json').exists():
            parent = candidate.parent
            if (parent / 'core').exists():
                return parent
    raise RuntimeError('No se pudo resolver PROJECT_ROOT: debe existir core/ y flujo_geosupport_final/settings.json')

PROJECT_ROOT = find_project_root()
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

FINAL_DIR = PROJECT_ROOT / 'flujo_geosupport_final'
SETTINGS_PATH = FINAL_DIR / 'settings.json'
SCRIPT_PATH = FINAL_DIR / 'scripts' / 'normalizar_rgb_streaming.py'
OUTPUT_DIR = FINAL_DIR / 'outputs' / '01_normalizar_rgb_streaming'
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

def resolve_path(value):
    if value in (None, ''):
        return None
    text = str(value)
    path = Path(text)
    if path.is_absolute() or text.startswith('\\'):
        return path
    return PROJECT_ROOT / path

with SETTINGS_PATH.open('r', encoding='utf-8') as file_obj:
    SETTINGS = json.load(file_obj)

## Parametros

`INPUT_FOLDER_TIFS` es el directorio raiz entregado por el cliente. El script excluye automaticamente cualquier subcarpeta llamada `normalizadas` para evitar reprocesar salidas anteriores.

In [ ]:
INPUT_FOLDER_TIFS = resolve_path(SETTINGS['input_folder_tifs'])
OUTPUT_FOLDER_NAME = SETTINGS.get('normalized_folder_name', 'normalizadas')

rasterio_env_local = resolve_path(SETTINGS.get('rasterio_env_local'))
rasterio_env_server = resolve_path(SETTINGS.get('rasterio_env_server'))
if rasterio_env_local and (rasterio_env_local / 'python.exe').exists():
    RASTERIO_PYTHON = rasterio_env_local / 'python.exe'
else:
    RASTERIO_PYTHON = rasterio_env_server / 'python.exe'

BACKGROUND_VALUES = SETTINGS.get('background_values')
EDGE_SIZE = int(SETTINGS.get('edge_size', 512))
TILE_SIZE = int(SETTINGS.get('tile_size', 512))

print('Settings:', SETTINGS_PATH)
print('Proyecto:', PROJECT_ROOT)
print('Entrada TIFF:', INPUT_FOLDER_TIFS)
print('Salida normalizada:', INPUT_FOLDER_TIFS / OUTPUT_FOLDER_NAME)
print('Python rasterio:', RASTERIO_PYTHON)
print('Script:', SCRIPT_PATH)

## Validaciones previas

In [ ]:
if not INPUT_FOLDER_TIFS.exists():
    raise FileNotFoundError(f'No existe INPUT_FOLDER_TIFS: {INPUT_FOLDER_TIFS}')
if not SCRIPT_PATH.exists():
    raise FileNotFoundError(f'No existe script de normalizacion: {SCRIPT_PATH}')
if not RASTERIO_PYTHON.exists():
    raise FileNotFoundError(f'No existe Python rasterio: {RASTERIO_PYTHON}')

source_tifs = []
for path in INPUT_FOLDER_TIFS.rglob('*'):
    if path.is_file() and path.suffix.lower() in {'.tif', '.tiff'}:
        if not any(part.lower() == OUTPUT_FOLDER_NAME.lower() for part in path.parts):
            source_tifs.append(path)
source_tifs = sorted(source_tifs, key=lambda item: str(item).lower())

print('TIFF detectados:', len(source_tifs))
for path in source_tifs[:20]:
    print(path)
if not source_tifs:
    raise RuntimeError('No se detectaron TIFF para normalizar.')

## Ejecutar normalizacion

In [ ]:
cmd = [
    str(RASTERIO_PYTHON),
    str(SCRIPT_PATH),
    str(INPUT_FOLDER_TIFS),
    '--output-folder-name', OUTPUT_FOLDER_NAME,
    '--edge-size', str(EDGE_SIZE),
    '--tile-size', str(TILE_SIZE),
]
if BACKGROUND_VALUES:
    cmd.extend(['--background-values', str(BACKGROUND_VALUES)])

def build_rasterio_subprocess_env(rasterio_python):
    env = os.environ.copy()
    env_prefix = Path(rasterio_python).resolve().parent
    dll_paths = [
        env_prefix / 'Library' / 'bin',
        env_prefix / 'Scripts',
        env_prefix / 'DLLs',
        env_prefix,
    ]
    base_path = [part for part in env.get('PATH', '').split(os.pathsep) if 'ArcGIS' not in part and 'arcgis' not in part.lower()]
    env['PATH'] = os.pathsep.join([str(path) for path in dll_paths if path.exists()] + base_path)
    gdal_data = env_prefix / 'Library' / 'share' / 'gdal'
    proj_lib = env_prefix / 'Library' / 'share' / 'proj'
    if gdal_data.exists():
        env['GDAL_DATA'] = str(gdal_data)
    if proj_lib.exists():
        env['PROJ_LIB'] = str(proj_lib)
    for key in list(env):
        if key.startswith('CONDA') or key in {'PYTHONPATH', 'PYTHONHOME', 'GDAL_DRIVER_PATH'}:
            env.pop(key, None)
    env['CONDA_PREFIX'] = str(env_prefix)
    env['CONDA_DEFAULT_ENV'] = str(env_prefix)
    env['CONDA_DLL_SEARCH_MODIFICATION_ENABLE'] = '1'
    return env

subprocess_env = build_rasterio_subprocess_env(RASTERIO_PYTHON)

print('Ejecutando:')
print(' '.join(f'"{part}"' if ' ' in part else part for part in cmd))
print('Python rasterio:', RASTERIO_PYTHON)
print('GDAL_DATA:', subprocess_env.get('GDAL_DATA'))
print('PROJ_LIB:', subprocess_env.get('PROJ_LIB'))

completed = subprocess.run(cmd, capture_output=True, text=True, env=subprocess_env)
(OUTPUT_DIR / 'stdout.log').write_text(completed.stdout or '', encoding='utf-8')
(OUTPUT_DIR / 'stderr.log').write_text(completed.stderr or '', encoding='utf-8')

print(completed.stdout[-6000:] if completed.stdout else '')
if completed.stderr:
    print('STDERR:')
    print(completed.stderr[-6000:])
if completed.returncode != 0:
    raise RuntimeError(f'Normalizacion fallo con codigo {completed.returncode}')

print('Normalizacion finalizada OK')
print('CSV resultados esperado:', INPUT_FOLDER_TIFS / OUTPUT_FOLDER_NAME / 'normalizacion_streaming_resultados.csv')
